In [1]:
!pip install groq python-dotenv numpy tqdm datasets math-verify

In [2]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset, concatenate_datasets

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any, Optional

load_dotenv()
random.seed(0)

client = Groq()

MODEL = "llama-3.1-8b-instant"

#### MATH 데이터셋 불러오기

- 평가: `HuggingFaceH4/MATH-500`
- few-shot 예시: `HuggingFaceH4/MATH`의 과목별 train split


In [3]:
# MATH 데이터 난이도 및 과목 필터링
TARGET_LEVELS = [1, 2, 3]

TARGET_SUBJECTS = [
    "Algebra",
    "Intermediate Algebra",
    "Number Theory",
    "Counting & Probability",
]

# 평가용 MATH-500
math_dataset = load_dataset("HuggingFaceH4/MATH-500")
math_test_raw = math_dataset["test"]

# few-shot 예시용 MATH train
TRAIN_CONFIGS = [
    "algebra",
    "intermediate_algebra",
    "number_theory",
    "counting_and_probability",
]

train_parts = []

for config in TRAIN_CONFIGS:
    ds = load_dataset(
        "HuggingFaceH4/MATH",
        config,
        split="train"
    )
    train_parts.append(ds)

math_train_raw = concatenate_datasets(train_parts)

print(sorted(set(math_train_raw["type"])))
print("raw train size:", len(math_train_raw))


['Algebra', 'Counting & Probability', 'Intermediate Algebra', 'Number Theory']
raw train size: 4679


In [4]:
## 데이터셋 전처리
def extract_last_boxed(text: str) -> Optional[str]:
    """문자열에서 마지막 \\boxed{...}의 내용을 추출합니다."""
    if not text:
        return None

    starts = [m.start() for m in re.finditer(r"\\boxed\s*\{", text)]
    if not starts:
        return None

    start = starts[-1]
    open_brace = text.find("{", start)
    depth = 0

    for idx in range(open_brace, len(text)):
        if text[idx] == "{":
            depth += 1
        elif text[idx] == "}":
            depth -= 1
            if depth == 0:
                return text[open_brace + 1:idx].strip()

    return None


def parse_level(level_value) -> Optional[int]:
    match = re.search(r"\d+", str(level_value))
    return int(match.group()) if match else None


def prepare_math_train_row(row):
    return {
        "question": row["problem"],
        "answer": extract_last_boxed(row["solution"]),
        "rationale": row["solution"],
        "subject": row["type"],
        "level_num": parse_level(row["level"]),
    }


math_train = math_train_raw.map(prepare_math_train_row)

math_train = math_train.filter(
    lambda row: (
        row["level_num"] in TARGET_LEVELS
        and row["subject"] in TARGET_SUBJECTS
        and row["answer"] is not None
    )
)

math_test = math_test_raw.filter(
    lambda row: (
        parse_level(row["level"]) in TARGET_LEVELS
        and row["subject"] in TARGET_SUBJECTS
    )
)

print("math_train size:", len(math_train))
print("math_test size:", len(math_test))
print("train levels:", sorted(set(math_train["level_num"])))
print("test levels:", sorted(set(parse_level(x) for x in math_test["level"])))


math_train size: 2132
math_test size: 146
train levels: [1, 2, 3]
test levels: [1, 2, 3]


In [5]:
def generate_response_using_Llama(
        prompt: str,
        model: str = MODEL
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.0,
            stream=False
        )
        return chat_completion.choices[0].message.content

    except Exception as e:
        print(f"API call error: {str(e)}")
        return None


#### 응답 잘 나오는지 확인하기

In [6]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello world! I'm here to help with any math problems you might have. What's on your mind? Do you have a specific problem you'd like me to solve, or would you like some help with a particular math concept?


#### MATH 데이터셋 확인하기

In [7]:
print("[Question]")
print(math_test[0]["problem"])
print("=" * 100)
print("[Answer]")
print(math_test[0]["answer"])
print("=" * 100)
print("[Solution]")
print(math_test[0]["solution"])


[Question]
If $f(x) = \frac{3x-2}{x-2}$, what is the value of $f(-2) +f(-1)+f(0)$? Express your answer as a common fraction.
[Answer]
\frac{14}{3}
[Solution]
$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3(-1)-2}{-1-2}+\frac{3(0)-2}{0-2}=\frac{-8}{-4}+\frac{-5}{-3}+\frac{-2}{-2}=2+\frac{5}{3}+1=\boxed{\frac{14}{3}}$


#### Utils 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [8]:
def extract_final_answer(response: str):
    """응답에서 마지막 \\boxed{...} 또는 Answer: 뒤의 답을 추출합니다."""
    if response is None:
        return None

    boxed_answer = extract_last_boxed(response)
    if boxed_answer is not None:
        return boxed_answer

    matches = re.findall(
        r"(?:Final Answer|Answer)\s*:\s*(.+)",
        response,
        re.IGNORECASE
    )
    if matches:
        return matches[-1].strip().strip("$")

    return None


def normalize_math_text(text: Any) -> str:
    text = str(text).strip().strip("$")
    text = text.replace(r"\displaystyle", "")
    text = text.replace(r"\dfrac", r"\frac")
    text = text.replace(r"\tfrac", r"\frac")
    text = text.replace(r"\,", "")
    text = text.replace(" ", "")
    return text.rstrip(".")


try:
    from math_verify import parse, verify
    MATH_VERIFY_AVAILABLE = True
except Exception:
    MATH_VERIFY_AVAILABLE = False


def answers_equivalent(
    correct_answer: str,
    predicted_answer: Optional[str]
) -> bool:
    if predicted_answer is None:
        return False

    if MATH_VERIFY_AVAILABLE:
        try:
            correct_parsed = parse(f"${correct_answer}$")
            predicted_parsed = parse(f"${predicted_answer}$")

            if verify(correct_parsed, predicted_parsed):
                return True
        except Exception:
            pass

    return (
        normalize_math_text(correct_answer)
        == normalize_math_text(predicted_answer)
    )


print("math-verify available:", MATH_VERIFY_AVAILABLE)


math-verify available: True


In [9]:
### 수정해도 됩니다!
import time

REQUEST_DELAY_SEC = 3.0


def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = MODEL,
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["problem"]
        correct_answer = str(dataset[i]["answer"]).strip()

        final_prompt = prompt.replace("{question}", question)

        response = generate_response_using_Llama(
            prompt=final_prompt,
            model=model
        )
        time.sleep(REQUEST_DELAY_SEC)  # rate limit 안전장치

        predicted_answer = (
            extract_final_answer(response)
            if response else None
        )
        is_correct = answers_equivalent(
            correct_answer,
            predicted_answer
        )

        if VERBOSE:
            print("=" * 50)
            print(response)
            print(f"Correct Answer: {correct_answer}")
            print(f"Predicted Answer: {predicted_answer}")
            print(f"Correct: {is_correct}")
            print("=" * 50)

        if is_correct:
            correct += 1

        total += 1

        results.append({
            "question": question,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct,
            "subject": dataset[i]["subject"],
            "level": parse_level(dataset[i]["level"]),
            "response": response,
        })

        if total % 5 == 0:
            current_accuracy = correct / total
            print(f"Progress: [{total}/{min(num_samples, len(dataset))}]")
            print(f"Current Acc.: [{current_accuracy:.2%}]")

    accuracy = correct / total if total > 0 else 0.0
    return results, accuracy

In [10]:
def save_final_result(
    results: List[Dict[str, Any]],
    accuracy: float,
    filename: str
) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += "[Details]\n"

    for idx, result in enumerate(results):
        result_str += f"Question {idx + 1}: {result['question']}\n"
        result_str += f"Subject: {result['subject']}\n"
        result_str += f"Level: {result['level']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)


#### 1. Direct Prompting with few-shot examples

In [11]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train

    sampled_indices = random.sample(
        range(len(train_dataset)),
        num_examples
    )

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question and generate ONLY the final answer "
        "after the tag 'Answer:' without any rationale. "
        "Use valid mathematical notation.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_answer = train_dataset[i]["answer"]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer: {cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt


In [12]:
### 어떤 방식으로 저장되는지 확인해보세요!

PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=math_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")
print(f"Direct 3-shot demo accuracy: {accuracy:.2%}")


 50%|█████     | 5/10 [00:17<00:17,  3.47s/it]

Progress: [5/10]
Current Acc.: [80.00%]


100%|██████████| 10/10 [00:35<00:00,  3.51s/it]

Progress: [10/10]
Current Acc.: [70.00%]
Direct 3-shot demo accuracy: 70.00%


In [13]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!
direct_accuracies = {}

for shot in [0, 3, 5]:
    prompt = construct_direct_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=math_test,
        prompt=prompt,
        num_samples=50
    )
    save_final_result(results, accuracy, f"direct_prompting_{shot}.txt")
    direct_accuracies[shot] = accuracy
    print(f"Direct {shot}-shot accuracy: {accuracy:.2%}")

print(direct_accuracies)

 10%|█         | 5/50 [00:16<02:30,  3.33s/it]

Progress: [5/50]
Current Acc.: [0.00%]


 20%|██        | 10/50 [00:33<02:12,  3.30s/it]

Progress: [10/50]
Current Acc.: [10.00%]


 30%|███       | 15/50 [00:52<02:05,  3.58s/it]

...

Progress: [40/50]
Current Acc.: [40.00%]


 90%|█████████ | 45/50 [03:25<00:18,  3.73s/it]

Progress: [45/50]
Current Acc.: [35.56%]


100%|██████████| 50/50 [04:06<00:00,  4.92s/it]

Progress: [50/50]
Current Acc.: [34.00%]
Direct 5-shot accuracy: 34.00%
{0: 0.16, 3: 0.54, 5: 0.34}


#### 2. Chain-of-Thought Prompting with few-shot examples

```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 됩니다.

In [13]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train

    sampled_indices = random.sample(
        range(len(train_dataset)),
        num_examples
    )

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question step by step. "
        "Show your reasoning clearly and explain each step, then give the final answer "
        "after the tag 'Answer:' wrapped in \\boxed{}. Use valid mathematical notation.\n"
    )

    for idx, i in enumerate(sampled_indices):
        # TODO: CoT 예시를 추가해주세요!
        cur_question = train_dataset[i]["question"]
        cur_rationale = train_dataset[i]["rationale"]  # 문제 풀이 과정 + \boxed{}로 끝나는 정답 포함

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer: {cur_rationale}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [15]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!
cot_accuracies = {}

for shot in [0, 3, 5]:
    prompt = construct_CoT_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=math_test,
        prompt=prompt,
        num_samples=50
    )
    save_final_result(results, accuracy, f"CoT_prompting_{shot}.txt")
    cot_accuracies[shot] = accuracy
    print(f"CoT {shot}-shot accuracy: {accuracy:.2%}")

print(cot_accuracies)

 10%|█         | 5/50 [00:18<02:49,  3.76s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:38<02:41,  4.04s/it]

Progress: [10/50]
Current Acc.: [70.00%]


 30%|███       | 15/50 [00:57<02:16,  3.91s/it]

...

Progress: [40/50]
Current Acc.: [65.00%]


 90%|█████████ | 45/50 [13:38<01:31, 18.21s/it]

Progress: [45/50]
Current Acc.: [64.44%]


100%|██████████| 50/50 [15:28<00:00, 18.58s/it]

Progress: [50/50]
Current Acc.: [64.00%]
CoT 5-shot accuracy: 64.00%
{0: 0.58, 3: 0.68, 5: 0.64}


#### 3. Construct your prompt + few shot examples
목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올리기!
- 세션때 배운 내용을 활용하거나 본인만의 풀이 과정을 만드는 등 자유롭게 진행해주시면 됩니다.
- 정답률은 Direct Prompting, CoT Prompting을 한 결과보다 높으면 됩니다. (0-shot, 3-shot, 5-shot 각각에서 모두 Direct Prompting과 CoT Prompting보다 높은 정답률을 달성하지 않더라도 감안하여 채점하겠습니다. 종합적으로 비교했을 때 본인이 설계한 프롬프트가 전반적으로 더 높은 성능을 보이는지를 기준으로 보겠습니다.)

In [14]:
def build_my_example(row) -> str:
    question = row["question"]
    rationale = row["rationale"]  # 이미 \boxed{}로 끝남
    return f"Question:\n{question}\nAnswer: {rationale}\n"

def construct_my_prompt(example_list, num_examples=3):
    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question step by step. "
        "Show your reasoning clearly and explain each step. Before finalizing, briefly double-check your calculation. "
        "Then give the final answer after the tag 'Answer:' wrapped in \\boxed{}. Use valid mathematical notation.\n"
    )
    for example in example_list[:num_examples]:
        prompt += f"\n[Example]\n{example}\n"
    prompt += "\nQuestion:\n{question}\nAnswer:"
    return prompt

In [15]:
from collections import defaultdict
subject_groups = defaultdict(list)
for i in range(len(math_train)):
    if len(math_train[i]["rationale"]) < 300:
        subject_groups[math_train[i]["subject"]].append(i)

subjects = list(subject_groups.keys())
random.shuffle(subjects)
my_example_indices = []
for subj in subjects:
    if len(my_example_indices) >= 5:
        break
    my_example_indices.append(random.choice(subject_groups[subj]))
if len(my_example_indices) < 5:
    remaining = [i for lst in subject_groups.values() for i in lst if i not in my_example_indices]
    my_example_indices += random.sample(remaining, 5 - len(my_example_indices))

my_examples = [build_my_example(math_train[i]) for i in my_example_indices]

my_accuracies = {}
for shot in [0, 3, 5]:
    prompt = construct_my_prompt(my_examples, num_examples=shot)
    results, accuracy = run_benchmark_test(dataset=math_test, prompt=prompt, num_samples=50)
    my_accuracies[shot] = accuracy
    save_final_result(results, accuracy, f"My_prompting_{shot}.txt")

print(my_accuracies)

 10%|█         | 5/50 [00:18<02:45,  3.68s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:37<02:36,  3.92s/it]

Progress: [10/50]
Current Acc.: [80.00%]


 30%|███       | 15/50 [01:19<05:51, 10.05s/it]

...

Progress: [40/50]
Current Acc.: [70.00%]


 90%|█████████ | 45/50 [09:08<00:51, 10.37s/it]

Progress: [45/50]
Current Acc.: [68.89%]


100%|██████████| 50/50 [10:29<00:00, 12.59s/it]

Progress: [50/50]
Current Acc.: [68.00%]
{0: 0.68, 3: 0.76, 5: 0.68}


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot 정답률을 표로 보여주세요.
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요.
3. 본인이 작성한 프롬프트 기법에 대해서 설명하고 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요.
4. 위 내용들을 `PROMPTING.md`에 보고서로 작성해주세요.
